# Laboratorio de Lógica Difusa (3 horas) — Colab
**Fecha:** 2025-08-26  
Basado en tu material *HACK.pdf* (33 diapositivas).

> Objetivo: Comprender y **demostrar en Python** propiedades y operaciones de la lógica difusa; construir y manipular **funciones de membresía**, **conjuntos difusos**, **relaciones difusas**, **producto cartesiano difuso** y **composición de relaciones**; cerrar con una mini-inferencia tipo Mamdani y defuzzificación.

## Agenda (180 minutos)
1. Propiedades de la lógica difusa (20 min)
2. Funciones de membresía: triangular, trapezoidal, gaussiana, sigmoide (30 min)
3. Conjuntos difusos y operaciones (unión, intersección, complemento) (25 min)
4. Propiedades de las operaciones difusas (15 min)
5. Relaciones difusas y producto cartesiano difuso (30 min)
6. Composición entre relaciones difusas (max–min, max–producto) (20 min)
7. Extra: α-cortes y defuzzificación (centroide). Mini sistema tipo Mamdani (30 min)
8. Cierre y checklist de aprendizaje (10 min)

## Configuración
Ejecuta esta celda primero.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1) Propiedades de la lógica difusa (demostrativas)
**Meta:** Comprobar propiedades clave con ejemplos numéricos.
> Conecta con tu teoría de **definiciones básicas y propiedades**. (Diapositivas intro/conceptos)

In [ ]:
# Universo discreto U y dos conjuntos difusos A, B
U = np.linspace(0, 10, 11)  # 0,1,2,...,10

# Definimos membresías con base triangular simple para ejemplo
def mu_A(x):
    # Triangular con pico en 3, soporte [0,6]
    a, b, c = 0, 3, 6
    if x <= a or x >= c: return 0.0
    if x == b: return 1.0
    if a < x < b: return (x - a) / (b - a)
    return (c - x) / (c - b)

def mu_B(x):
    # Triangular con pico en 7, soporte [4,10]
    a, b, c = 4, 7, 10
    if x <= a or x >= c: return 0.0
    if x == b: return 1.0
    if a < x < b: return (x - a) / (b - a)
    return (c - x) / (c - b)

A = np.array([mu_A(x) for x in U])
B = np.array([mu_B(x) for x in U])

print("U:", U)
print("A:", A)
print("B:", B)

### Operaciones básicas (min, max, complemento)

In [ ]:
def complemento(A):
    A = np.asarray(A, dtype=float)
    return 1.0 - A

def interseccion(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return np.minimum(A, B)

def union(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    return np.maximum(A, B)

# Variantes (t-norm y s-norm comunes)
def interseccion_producto(A, B):
    return np.asarray(A) * np.asarray(B)

def union_prob_suma(A, B):
    A = np.asarray(A); B = np.asarray(B)
    return A + B - A*B

### Visualización de A y B

In [ ]:
plt.figure()
plt.plot(U, A, marker='o', label='A')
plt.plot(U, B, marker='s', label='B')
plt.title("Conjuntos difusos A y B sobre U")
plt.xlabel("U")
plt.ylabel("μ")
plt.legend()
plt.show()

### Comprobaciones de propiedades
- Conmutatividad: \(A \cup B = B \cup A\), \(A \cap B = B \cap A\)
- Idempotencia: \(A \cup A = A\), \(A \cap A = A\)
- Absorción: \(A \cup (A \cap B) = A\), \(A \cap (A \cup B) = A\)
- De Morgan: \(\overline{A \cup B} = \overline{A} \cap \overline{B}\), \(\overline{A \cap B} = \overline{A} \cup \overline{B}\)

In [ ]:
def casi_igual(X, Y, eps=1e-9):
    return np.all(np.abs(np.asarray(X) - np.asarray(Y)) < eps)

# Conmutatividad
assert casi_igual(union(A,B), union(B,A))
assert casi_igual(interseccion(A,B), interseccion(B,A))

# Idempotencia
assert casi_igual(union(A,A), A)
assert casi_igual(interseccion(A,A), A)

# Absorción
assert casi_igual(union(A, interseccion(A,B)), A)
assert casi_igual(interseccion(A, union(A,B)), A)

# De Morgan
lhs1 = complemento(union(A,B))
rhs1 = interseccion(complemento(A), complemento(B))
lhs2 = complemento(interseccion(A,B))
rhs2 = union(complemento(A), complemento(B))

assert casi_igual(lhs1, rhs1)
assert casi_igual(lhs2, rhs2)

print("✅ Propiedades básicas comprobadas numéricamente")

## 2) Funciones de membresía
**Meta:** Implementar y visualizar funciones **triangular, trapezoidal, gaussiana y sigmoide**.

In [ ]:
def triangular(x, a, b, c):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    asc = (x >= a) & (x < b)
    desc = (x > b) & (x <= c)
    y[asc] = (x[asc] - a) / (b - a) if b != a else 0
    y[desc] = (c - x[desc]) / (c - b) if c != b else 0
    y[x == b] = 1.0
    return np.clip(y, 0, 1)

def trapezoidal(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    asc = (x >= a) & (x < b)
    top = (x >= b) & (x <= c)
    desc = (x > c) & (x <= d)
    y[asc] = (x[asc] - a) / (b - a) if b != a else 0
    y[top] = 1.0
    y[desc] = (d - x[desc]) / (d - c) if d != c else 0
    return np.clip(y, 0, 1)

def gaussiana(x, c, sigma):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5*((x - c)/sigma)**2)

def sigmoide(x, c, a):
    x = np.asarray(x, dtype=float)
    return 1/(1 + np.exp(-a*(x - c)))

In [ ]:
x = np.linspace(0, 10, 200)

plt.figure()
plt.plot(x, triangular(x, 2, 5, 8))
plt.title("Triangular(2,5,8)")
plt.xlabel("x"); plt.ylabel("μ")
plt.show()

plt.figure()
plt.plot(x, trapezoidal(x, 1, 3, 7, 9))
plt.title("Trapezoidal(1,3,7,9)")
plt.xlabel("x"); plt.ylabel("μ")
plt.show()

plt.figure()
plt.plot(x, gaussiana(x, c=5, sigma=1.5))
plt.title("Gaussiana(c=5, sigma=1.5)")
plt.xlabel("x"); plt.ylabel("μ")
plt.show()

plt.figure()
plt.plot(x, sigmoide(x, c=5, a=1.0))
plt.title("Sigmoide(c=5, a=1.0)")
plt.xlabel("x"); plt.ylabel("μ")
plt.show()

In [ ]:
# Tests simples de membresía en puntos clave
assert triangular([2,5,8], 2,5,8).tolist() == [0.0, 1.0, 0.0]
trap = trapezoidal([1,3,7,9], 1,3,7,9).tolist()
assert trap == [0.0, 1.0, 1.0, 0.0]
g = gaussiana([5], 5, 1.0)[0]
assert abs(g - 1.0) < 1e-9
s = sigmoide([5], 5, 100.0)[0]
assert s > 0.99
print("✅ Funciones de membresía OK")

## 3) Conjuntos difusos sobre un universo U
**Meta:** Definir etiquetas lingüísticas (Bajo, Medio, Alto) para calificaciones/temperatura/etc.

In [ ]:
U_cont = np.linspace(0, 100, 201)

Bajo = trapezoidal(U_cont, 0, 0, 20, 40)
Medio = triangular(U_cont, 30, 50, 70)
Alto = trapezoidal(U_cont, 60, 80, 100, 100)

plt.figure()
plt.plot(U_cont, Bajo, label="Bajo")
plt.plot(U_cont, Medio, label="Medio")
plt.plot(U_cont, Alto, label="Alto")
plt.title("Etiquetas lingüísticas sobre U=[0,100]")
plt.xlabel("U"); plt.ylabel("μ")
plt.legend()
plt.show()

In [ ]:
def fuzzificar(valor, U, etiquetas):
    # etiquetas: dict nombre -> membresía(U)
    # devuelve dict nombre -> μ(valor)
    # Se interpola sobre U_cont
    mu = {}
    for nombre, mu_U in etiquetas.items():
        idx = np.abs(U - valor).argmin()
        mu[nombre] = float(mu_U[idx])
    return mu

etiquetas = {"Bajo": Bajo, "Medio": Medio, "Alto": Alto}
ej = fuzzificar(65, U_cont, etiquetas)
print("Fuzzificación de 65:", ej)
assert 0 <= ej["Bajo"] <= 1 and 0 <= ej["Medio"] <= 1 and 0 <= ej["Alto"] <= 1
print("✅ Fuzzificación OK")

## 4) Operaciones entre conjuntos difusos
**Meta:** Implementar unión, intersección y complemento; comparar min/max vs. producto/suma probabilística.

In [ ]:
C = triangular(U_cont, 20, 50, 80)
D = trapezoidal(U_cont, 40, 55, 75, 90)

Uni_minmax = np.maximum(C, D)
Int_minmax = np.minimum(C, D)
Comp_C = 1 - C

Uni_prob = D + C - D*C
Int_prod = D * C

plt.figure()
plt.plot(U_cont, C, label="C")
plt.plot(U_cont, D, label="D")
plt.title("Conjuntos C y D")
plt.xlabel("U"); plt.ylabel("μ")
plt.legend(); plt.show()

plt.figure()
plt.plot(U_cont, Uni_minmax, label="Unión (max)")
plt.plot(U_cont, Int_minmax, label="Intersección (min)")
plt.plot(U_cont, Comp_C, label="Complemento(C)")
plt.title("Operaciones (min/max)")
plt.xlabel("U"); plt.ylabel("μ")
plt.legend(); plt.show()

plt.figure()
plt.plot(U_cont, Uni_prob, label="Unión (suma prob.)")
plt.plot(U_cont, Int_prod, label="Intersección (producto)")
plt.title("Operaciones alternativas")
plt.xlabel("U"); plt.ylabel("μ")
plt.legend(); plt.show()

## 5) Propiedades de las operaciones difusas (tests)
**Meta:** Validar con asserts: conmutatividad, asociatividad, distributividad, De Morgan.

In [ ]:
def approx_eq(X, Y, tol=1e-9): return np.allclose(X, Y, atol=tol, rtol=0)

# Conmutatividad
assert approx_eq(np.maximum(C,D), np.maximum(D,C))
assert approx_eq(np.minimum(C,D), np.minimum(D,C))

# Asociatividad
E = triangular(U_cont, 0, 25, 50)
assert approx_eq(np.maximum(np.maximum(C,D),E), np.maximum(C, np.maximum(D,E)))
assert approx_eq(np.minimum(np.minimum(C,D),E), np.minimum(C, np.minimum(D,E)))

# Distributividad
lhs1 = np.minimum(C, np.maximum(D,E))
rhs1 = np.maximum(np.minimum(C,D), np.minimum(C,E))
lhs2 = np.maximum(C, np.minimum(D,E))
rhs2 = np.minimum(np.maximum(C,D), np.maximum(C,E))
assert approx_eq(lhs1, rhs1)
assert approx_eq(lhs2, rhs2)

# De Morgan
assert approx_eq(1 - np.maximum(C,D), np.minimum(1-C, 1-D))
assert approx_eq(1 - np.minimum(C,D), np.maximum(1-C, 1-D))

print("✅ Propiedades de operaciones comprobadas")

## 6) Relaciones difusas
**Meta:** Crear una **relación difusa R** en U×V y visualizarla como matriz.

In [ ]:
U_rel = np.linspace(0, 1, 51)
V_rel = np.linspace(0, 1, 51)

# Relación "cercanía": alta membresía cuando u ≈ v
sigma = 0.15
R = np.zeros((len(U_rel), len(V_rel)))
for i,u in enumerate(U_rel):
    for j,v in enumerate(V_rel):
        R[i,j] = math.exp(-0.5*((u - v)/sigma)**2)

plt.figure()
plt.imshow(R, origin='lower', extent=[V_rel.min(), V_rel.max(), U_rel.min(), U_rel.max()], aspect='auto')
plt.title("Relación difusa R (cercanía)")
plt.xlabel("V"); plt.ylabel("U")
plt.colorbar()
plt.show()

print("R shape:", R.shape, "μ in [", R.min(), ",", R.max(), "]")

## 7) Producto cartesiano difuso A × B
**Meta:** Dado A⊆U y B⊆V, construir \( (A×B)(u,v) = \min\{A(u),B(v)\} \).

In [ ]:
# A_f y B_f sobre U_rel, V_rel
A_f = sigmoide(U_rel, c=0.5, a=10)
B_f = sigmoide(V_rel, c=0.5, a=-10)  # Sigmoide decreciente

AxB = np.zeros((len(U_rel), len(V_rel)))
for i,u in enumerate(U_rel):
    for j,v in enumerate(V_rel):
        AxB[i,j] = min(A_f[i], B_f[j])

plt.figure()
plt.imshow(AxB, origin='lower', extent=[V_rel.min(), V_rel.max(), U_rel.min(), U_rel.max()], aspect='auto')
plt.title("Producto cartesiano difuso A×B (min)")
plt.xlabel("V"); plt.ylabel("U")
plt.colorbar()
plt.show()

print("A×B shape:", AxB.shape, "μ in [", AxB.min(), ",", AxB.max(), "]")

## 8) Composición entre relaciones difusas
**Meta:** Implementar **max–min** y **max–producto** para componer R (U×V) y S (V×W).

In [ ]:
# Definimos S en V×W
W_rel = np.linspace(0, 1, 51)
tau = 0.2
S = np.zeros((len(V_rel), len(W_rel)))
for j,v in enumerate(V_rel):
    for k,w in enumerate(W_rel):
        S[j,k] = math.exp(-0.5*((v - (1-w))/tau)**2)  # Relación "anti-diagonal"

def comp_max_min(R, S):
    # R: U×V, S: V×W => T: U×W
    U_len, V_len = R.shape
    V_len2, W_len = S.shape
    assert V_len == V_len2
    T = np.zeros((U_len, W_len))
    for i in range(U_len):
        for k in range(W_len):
            T[i,k] = np.max(np.minimum(R[i,:], S[:,k]))
    return T

def comp_max_prod(R, S):
    U_len, V_len = R.shape
    V_len2, W_len = S.shape
    assert V_len == V_len2
    T = np.zeros((U_len, W_len))
    for i in range(U_len):
        for k in range(W_len):
            T[i,k] = np.max(R[i,:] * S[:,k])
    return T

T_min = comp_max_min(R, S)
T_prod = comp_max_prod(R, S)

plt.figure()
plt.imshow(T_min, origin='lower', extent=[W_rel.min(), W_rel.max(), U_rel.min(), U_rel.max()], aspect='auto')
plt.title("Composición R∘S — max–min")
plt.xlabel("W"); plt.ylabel("U")
plt.colorbar()
plt.show()

plt.figure()
plt.imshow(T_prod, origin='lower', extent=[W_rel.min(), W_rel.max(), U_rel.min(), U_rel.max()], aspect='auto')
plt.title("Composición R∘S — max–producto")
plt.xlabel("W"); plt.ylabel("U")
plt.colorbar()
plt.show()

print("T_min shape:", T_min.shape, "T_prod shape:", T_prod.shape)

In [ ]:
# Tests de composición: cota superior por 1 y no negatividad
assert T_min.min() >= -1e-12 and T_min.max() <= 1 + 1e-12
assert T_prod.min() >= -1e-12 and T_prod.max() <= 1 + 1e-12
print("✅ Composición difusa OK")

## 9) Extra — α-cortes y defuzzificación (centroide)
**Meta:** Obtener α-cortes de un conjunto y defuzzificar una salida agregada.

In [ ]:
def alpha_corte(U, mu, alpha):
    U = np.asarray(U); mu = np.asarray(mu)
    idx = mu >= alpha
    return U[idx]

# Conjunto de salida "Velocidad del ventilador"
V_out = np.linspace(0, 100, 201)
Lento = trapezoidal(V_out, 0, 0, 20, 40)
Medio_v = triangular(V_out, 30, 50, 70)
Rapido = trapezoidal(V_out, 60, 80, 100, 100)

# Agregación de salida (ejemplo simple)
agg = np.maximum(Rapido*0.6, np.maximum(Medio_v*0.3, Lento*0.1))

plt.figure()
plt.plot(V_out, Lento, label="Lento")
plt.plot(V_out, Medio_v, label="Medio")
plt.plot(V_out, Rapido, label="Rápido")
plt.plot(V_out, agg, label="Agregado", linewidth=2)
plt.title("Conjunto de salida y agregación")
plt.xlabel("V"); plt.ylabel("μ")
plt.legend(); plt.show()

def centroide(U, mu):
    U = np.asarray(U, dtype=float)
    mu = np.asarray(mu, dtype=float)
    num = np.trapz(U * mu, U)
    den = np.trapz(mu, U)
    return float(num/den) if den > 0 else float('nan')

print("α-corte alpha=0.5 (Lento):", alpha_corte(V_out, Lento, 0.5)[:5], "...")
print("Centroide(agg):", round(centroide(V_out, agg), 2))

## 10) Mini-inferencia tipo Mamdani (1 entrada → 1 salida)
**Caso:** Control de ventilador según **temperatura** (entrada) → **velocidad** (salida).  
Reglas:
1. Si Temp es **Baja** ⇒ Vel es **Lento**
2. Si Temp es **Media** ⇒ Vel es **Medio**
3. Si Temp es **Alta** ⇒ Vel es **Rápido**

In [ ]:
# Reutilizamos etiquetas "Bajo/Medio/Alto" para temperatura
Temp = U_cont  # 0..100
Temp_Bajo = Bajo
Temp_Medio = Medio
Temp_Alto = Alto

Vel = V_out
Vel_Lento = Lento
Vel_Medio = Medio_v
Vel_Rapido = Rapido

def mamdani_velocidad(temp_val):
    mu_in = fuzzificar(temp_val, Temp, {"Bajo": Temp_Bajo, "Medio": Temp_Medio, "Alto": Temp_Alto})
    r1 = min(mu_in["Bajo"], 1.0)   # Activa Lento
    r2 = min(mu_in["Medio"], 1.0)  # Activa Medio
    r3 = min(mu_in["Alto"], 1.0)   # Activa Rápido

    # Implicación (min) y agregación (max)
    out_lento = np.minimum(Vel_Lento, r1)
    out_medio = np.minimum(Vel_Medio, r2)
    out_rapido = np.minimum(Vel_Rapido, r3)
    agregado = np.maximum(out_rapido, np.maximum(out_medio, out_lento))
    salida = centroide(Vel, agregado)
    return mu_in, agregado, salida

mu_in, agregado, salida = mamdani_velocidad(65)

plt.figure()
plt.plot(Temp, Temp_Bajo, label="Temp Baja")
plt.plot(Temp, Temp_Medio, label="Temp Media")
plt.plot(Temp, Temp_Alto, label="Temp Alta")
plt.title("Entrada: Temperatura (etiquetas)")
plt.xlabel("Temperatura"); plt.ylabel("μ")
plt.legend(); plt.show()

plt.figure()
plt.plot(Vel, Vel_Lento, label="Lento")
plt.plot(Vel, Vel_Medio, label="Medio")
plt.plot(Vel, Vel_Rapido, label="Rápido")
plt.plot(Vel, agregado, label="Agregado")
plt.title("Salida: Velocidad (agregada)")
plt.xlabel("Velocidad"); plt.ylabel("μ")
plt.legend(); plt.show()

print("Fuzzificación(65):", mu_in)
print("Salida defuzzificada (centroide):", round(salida,2))

## Checklist de aprendizaje
- [ ] Comprendí y probé propiedades básicas y de operaciones difusas.
- [ ] Implementé funciones de membresía y ajusté parámetros.
- [ ] Construí y visualicé relaciones y productos cartesianos difusos.
- [ ] Apliqué composición max–min y max–producto.
- [ ] Realicé α-cortes y defuzzificación por centroide.
- [ ] Ejecuté una mini-inferencia Mamdani de punta a punta.